In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "api").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Ensure the project root and parent are on PYTHONPATH for Jupyter notebooks.
for candidate in (project_root, project_root.parent):
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        
try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

env_path = Path(project_root) / ".env"
if load_dotenv is not None and env_path.exists():
    load_dotenv(dotenv_path=env_path)

In [2]:
from api.core.db import SessionLocal
from sqlalchemy import text
from sqlalchemy import select
from sqlalchemy.orm import selectinload
from models.work_metadata.work import Work


In [7]:
from models.work_metadata.workAgent import WorkAgent
from models.work_metadata.workSubject import WorkSubject
from models.instance import Instance
from uuid import UUID


async with SessionLocal() as session:
    work_id = 'ea2493a8-0d5e-487e-90b3-13615ce84fa6'
    work_uuid = UUID(work_id)
        
    # result = await session.execute(
    #     select(Work)
    #     .where(Work.id == work_uuid)
    # )
    
    result = await session.execute(
                select(Work)
                .where(Work.id == work_uuid)
                .options(
                # Agents
                selectinload(Work.agents)
                    .selectinload(WorkAgent.agent),
    
                # Subjects
                selectinload(Work.subjects)
                    .selectinload(WorkSubject.subject),
    
                # Instances
                selectinload(Work.instances)
                    .selectinload(Instance.items),
    
                # Languages
                selectinload(Work.languages),
    
                # Identifiers
                selectinload(Work.identifiers),
    
                # Genres
                selectinload(Work.genres),
    
                # Titles
                selectinload(Work.titles),
    
                # Notes
                selectinload(Work.notes),
    
                # Relations
                # selectinload(Work.relations),
    
                # Types
                selectinload(Work.types),
            )
            )
  
    work = result.scalar_one_or_none()
    print("RES:", work)

2026-09-21 16:50:01,450 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-21 16:50:01,454 INFO sqlalchemy.engine.Engine SELECT work.id, work.title, work.summary, work.uri 
FROM work 
WHERE work.id = %s
2026-09-21 16:50:01,456 INFO sqlalchemy.engine.Engine [cached since 242s ago] ('ea2493a80d5e487e90b313615ce84fa6',)
2026-09-21 16:50:01,465 INFO sqlalchemy.engine.Engine SELECT work_relation.source_work_id AS work_relation_source_work_id, work_relation.id AS work_relation_id, work_relation.target_work_id AS work_relation_target_work_id, work_relation.relation_type AS work_relation_relation_type 
FROM work_relation 
WHERE work_relation.source_work_id IN (%s)
2026-09-21 16:50:01,468 INFO sqlalchemy.engine.Engine [cached since 242s ago] ('ea2493a80d5e487e90b313615ce84fa6',)
2026-09-21 16:50:01,475 INFO sqlalchemy.engine.Engine SELECT work_identifier.work_id AS work_identifier_work_id, work_identifier.id AS work_identifier_id, work_identifier.type AS work_identifier_type, work_identifie

In [8]:
instance = work.instances[0]
instance.items

[<models.item.Item at 0x7493580ad090>, <models.item.Item at 0x7493580ad1d0>]

In [10]:
item = instance.items[1]
item.uri

In [5]:
from api.indexer.mappers.work import WorkMapper


document = WorkMapper.to_search_document(work)

In [6]:
instance = document.instances[0]
instance

InstanceSummary(id=UUID('37844b26-8af5-41a1-9e1b-d1213a85106b'), isbn='978-6556403830', publication_year=2025, formato=None, publisher_id=UUID('e235df0a-fde4-4f04-9400-d74f94118b2f'), items=[ItemSearchDocument(id=UUID('d3257b85-3176-4f9e-86b1-37a348a93407'), instance_id=UUID('37844b26-8af5-41a1-9e1b-d1213a85106b'), uri=None, barcode='26-002', location=None, call_number=None, status='disponível'), ItemSearchDocument(id=UUID('32a8f71a-7fea-4dbd-8b3a-7ef81027f414'), instance_id=UUID('37844b26-8af5-41a1-9e1b-d1213a85106b'), uri=None, barcode='26-004', location=None, call_number=None, status='disponível'), ItemSearchDocument(id=UUID('a69cedd9-06d9-4ec7-b2a4-b55aadda7e92'), instance_id=UUID('37844b26-8af5-41a1-9e1b-d1213a85106b'), uri=None, barcode='26-005', location=None, call_number=None, status='disponível')])

In [9]:
instance.format

AttributeError: 'InstanceSummary' object has no attribute 'format'

In [53]:
for instance in work.instances:
    print(instance.items)


[<models.item.Item object at 0x74b52bf417f0>, <models.item.Item object at 0x74b52b41efd0>, <models.item.Item object at 0x74b52b41f110>]


In [ ]:
from api.indexer.documents.item import ItemSearchDocument
from api.indexer.documents.work import InstanceSummary


instances = [
                InstanceSummary(
                    id=instance.id,
                    isbn=instance.isbn,
                    publication_year=instance.publication_year,
                    format=instance.formato,
                    publisher_id=instance.publisher_id,
                    
                    items=[
                    ItemSearchDocument(
                        id=item.id,
                        instance_id=item.instance_id,
                        barcode=item.barcode,
                        status=item.status,
                    )
                    for item in instance.items
                ],
                    
                )
                for instance in work.instances
            ]

In [71]:
instances[0].items

AttributeError: 'InstanceSummary' object has no attribute 'items'

In [67]:
for instance in work.instances:
    for item in instance.items:
        x = ItemSearchDocument(
                                id=item.id,
                                instance_id=item.instance_id,
                                barcode=item.barcode,
                                status=item.status,
                            )
    
        print(x)

id=UUID('d3257b85-3176-4f9e-86b1-37a348a93407') instance_id=UUID('37844b26-8af5-41a1-9e1b-d1213a85106b') uri=None barcode='26-002' location=None call_number=None status='disponível'
id=UUID('32a8f71a-7fea-4dbd-8b3a-7ef81027f414') instance_id=UUID('37844b26-8af5-41a1-9e1b-d1213a85106b') uri=None barcode='26-004' location=None call_number=None status='disponível'
id=UUID('a69cedd9-06d9-4ec7-b2a4-b55aadda7e92') instance_id=UUID('37844b26-8af5-41a1-9e1b-d1213a85106b') uri=None barcode='26-005' location=None call_number=None status='disponível'
